<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML321ENSkillsNetwork817-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Extract Bag of Words (BoW) Features from Course Textual Content**


The main goal of recommender systems is to help users find items they potentially interested in. Depending on the recommendation tasks, an item can be a movie, a restaurant, or, in our case, an online course. 

Machine learning algorithms cannot work on an item directly so we first need to extract features and represent the items mathematically, i.e., with a feature vector.

Many items are often described by text so they are associated with textual data, such as the titles and descriptions of a movie or course. Since machine learning algorithms can not process textual data directly, we need to transform the raw text into numeric feature vectors.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/module_2/images/extract_textual_features.png)


## Objectives


After completing this lab you will be able to:


* Extract Bag of Words (BoW) features from course titles and descriptions
* Build a course BoW dataset to be used for building a content-based recommender system later


----


## Prepare and setup the lab environment


First, let's install and import required libraries:


In [1]:
import gensim
import pandas as pd
import nltk as nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim import corpora

%matplotlib inline

Download stopwords


In [21]:
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ferna\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ferna\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\ferna\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


True

### Extrac BoW features for course textual content and build a dataset

In [23]:
course_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_processed.csv"
course_content_df = pd.read_csv(course_url)

In [24]:
course_content_df.head(1)

,COURSE_ID,TITLE,DESCRIPTION
0,ML0201EN,robots are coming build iot apps with watson ...,have fun with iot and learn along the way if ...


The course content dataset has three columns `COURSE_ID`, `TITLE`, and `DESCRIPTION`. `TITLE` and `DESCRIPTION` are all text upon which we want to extract BoW features. 


Let's join those two text columns together.


In [25]:
# Merge TITLE and DESCRIPTION title
course_content_df['course_texts'] = course_content_df[['TITLE', 'DESCRIPTION']].agg(' '.join, axis=1)
course_content_df = course_content_df.reset_index()
course_content_df['index'] = course_content_df.index

In [26]:
course_content_df.iloc[0, :]

index                                                           0
COURSE_ID                                                ML0201EN
TITLE           robots are coming  build iot apps with watson ...
DESCRIPTION     have fun with iot and learn along the way  if ...
course_texts    robots are coming  build iot apps with watson ...
Name: 0, dtype: object

We have used the `tokenize_course()` method  to tokenize the course content:


In [27]:
def tokenize_course(course, keep_only_nouns=True):
    # Get English stop words
    stop_words = set(stopwords.words('english'))
    # Tokenize the course text
    word_tokens = word_tokenize(course)
    # Remove English stop words and numbers
    word_tokens = [w for w in word_tokens if (not w.lower() in stop_words) and (not w.isnumeric())]
    # Only keep nouns 
    if keep_only_nouns:
        # Define a filter list of non-noun POS tags
        filter_list = ['WDT', 'WP', 'WRB', 'FW', 'IN', 'JJR', 'JJS', 'MD', 'PDT', 'POS', 'PRP', 'RB', 'RBR', 'RBS',
                       'RP']
        # Tag the word tokens with POS tags
        tags = nltk.pos_tag(word_tokens)
        # Filter out non-nouns based on POS tags
        word_tokens = [word for word, pos in tags if pos not in filter_list]

    return word_tokens

Generate the BoW features for each course.

In [34]:
# Tokenizar todos los cursos usando la función tokenize_course
course_content_df['tokens'] = course_content_df['course_texts'].apply(lambda x: tokenize_course(x))
course_content_df[['COURSE_ID', 'tokens']].head()

,COURSE_ID,tokens
0,ML0201EN,"[robots, coming, build, iot, apps, watson, swi..."
1,ML0122EN,"[accelerating, deep, learning, gpu, training, ..."
2,GPXX0ZG0EN,"[consuming, restful, services, using, reactive..."
3,RP0105EN,"[analyzing, big, data, r, using, apache, spark..."
4,GPXX0Z2PEN,"[containerizing, packaging, running, spring, b..."


Create a token dictionary `tokens_dict`


In [35]:
# Crear un diccionario de tokens con gensim
tokens_dict = corpora.Dictionary(course_content_df['tokens'])
tokens_dict

Use `doc2bow()` method to generate BoW features for each tokenized course.


In [36]:
# Generar características BoW para cada curso
course_content_df['bow'] = course_content_df['tokens'].apply(lambda x: tokens_dict.doc2bow(x))
course_content_df[['COURSE_ID', 'bow']].head()

,COURSE_ID,bow
0,ML0201EN,"[(0, 2), (1, 2), (2, 2), (3, 1), (4, 1), (5, 1..."
1,ML0122EN,"[(0, 2), (3, 2), (6, 1), (12, 1), (30, 4), (34..."
2,GPXX0ZG0EN,"[(12, 1), (26, 1), (30, 1), (108, 2), (109, 1)..."
3,RP0105EN,"[(6, 4), (63, 1), (77, 1), (83, 1), (117, 1), ..."
4,GPXX0Z2PEN,"[(12, 1), (140, 2), (141, 2), (142, 1), (143, ..."


Append the BoW features for each course into a new BoW dataframe. The new dataframe needs to include the following columns:
- 'doc_index': the course index starting from 0
- 'doc_id': the actual course id such as `ML0201EN`
- 'token': the tokens for each course
- 'bow': the bow value for each token


In [37]:
# Crear listas vacías para cada columna
doc_indices = []
doc_ids = []
tokens_list = []
bow_values = []

# Iterar sobre cada curso y su BoW
for idx, row in course_content_df.iterrows():
    course_id = row['COURSE_ID']
    bow = row['bow']
    
    for token_id, count in bow:
        doc_indices.append(idx)
        doc_ids.append(course_id)
        tokens_list.append(tokens_dict[token_id])  # Usar tokens_dict para obtener el token correcto
        bow_values.append(count)

# Crear el nuevo DataFrame
course_bow_df = pd.DataFrame({
    'doc_index': doc_indices,
    'doc_id': doc_ids,
    'token': tokens_list,
    'bow': bow_values
})

course_bow_df.head()



,doc_index,doc_id,token,bow
0,0,ML0201EN,ai,2
1,0,ML0201EN,apps,2
2,0,ML0201EN,build,2
3,0,ML0201EN,cloud,1
4,0,ML0201EN,coming,1
